## Attention U-Net++ (ResNet-50) — notebook guide

This notebook trains **Attention U-Net++** (`build_resnet50_attention_unet_plus_plus`) for **camouflaged military object segmentation** on ACD1K-style data (see `docs/DATASET.md`).

- **Optimizer / LR:** matches the `att-unet++-adam-lr=1e-5-droput=0,2-clahesiz` naming (**Adam**, **lr = 1e-5**, dropout **0.2** in the hyperparameter cell).
- **CLAHE:** absent in `*clahesiz*` notebooks; enable the LAB–CLAHE block from a CLAHE notebook if you need that preprocessing path.
- **Documentation:** `docs/MODEL_ARCHITECTURE.md`, `docs/TRAINING.md`, `docs/EVALUATION_METRICS.md`.


In [ ]:
import os
import tensorflow as tf
import warnings

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

from tensorflow import keras as krs
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.applications import ResNet50
import tensorflow.keras.backend as K
import albumentations as A

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_recall_fscore_support,
    f1_score,
    precision_score,
    recall_score,
    roc_curve,
    auc
)
from tqdm import tqdm
import time

In [ ]:
# Path definitions
#PATH="Adaptive Camouflaged Dataset (ACD1K)/"
PATH="../input/dataset-splitM/"
TRAIN_IMG_PATH = PATH+"Training/images"
TRAIN_MASK_PATH = PATH+"Training/GT"
TEST_IMG_PATH =  PATH+"Testing/images"
TEST_MASK_PATH = PATH+"Testing/GT"

# Debug prints
print("Checking if directories exist:")
print(f"Training images: {os.path.exists(TRAIN_IMG_PATH)}")
print(f"Training masks: {os.path.exists(TRAIN_MASK_PATH)}")
print(f"Testing images: {os.path.exists(TEST_IMG_PATH)}")
print(f"Testing masks: {os.path.exists(TEST_MASK_PATH)}")

if os.path.exists(TRAIN_IMG_PATH):
    print("\nFirst 5 files in training images directory:")
    print(os.listdir(TRAIN_IMG_PATH)[:5])  # First 5 files

# Create output directories

os.makedirs('Project_Outputs/training_plots', exist_ok=True)
os.makedirs('Project_Outputs/model_outputs', exist_ok=True)
os.makedirs('Project_Outputs/logs', exist_ok=True)

#training parameters
IMAGE_HEIGHT=256
IMAGE_WIDTH=256
BATCH_SIZE=32
EPOCHS=200
KFOLD_NUM=10
LR=0.00001
DROPOUT_RATE=0.2

IMAGE_SIZE=(IMAGE_HEIGHT,IMAGE_WIDTH)

SEED=42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# Check directories and file structure

In [ ]:
print("Checking directories...")
print(f"Training images directory exists: {os.path.exists(TRAIN_IMG_PATH)}")
print(f"Training masks directory exists: {os.path.exists(TRAIN_MASK_PATH)}")
print(f"Testing images directory exists: {os.path.exists(TEST_IMG_PATH)}")
print(f"Testing masks directory exists: {os.path.exists(TEST_MASK_PATH)}")

def check_dataset_structure():
    """Check dataset structure and print detailed information"""
    print("\nChecking Dataset Structure:")
    
    # Training set
    print("\nTraining Set:")
    train_imgs = sorted(os.listdir(TRAIN_IMG_PATH))
    train_masks = sorted(os.listdir(TRAIN_MASK_PATH))
    print(f"Number of training images: {len(train_imgs)}")
    print(f"Number of training masks: {len(train_masks)}")
    print("\nFirst 5 training images:", train_imgs[:5])
    print("First 5 training masks:", train_masks[:5])
    
    # Testing set
    print("\nTesting Set:")
    test_imgs = sorted(os.listdir(TEST_IMG_PATH))
    test_masks = sorted(os.listdir(TEST_MASK_PATH))
    print(f"Number of testing images: {len(test_imgs)}")
    print(f"Number of testing masks: {len(test_masks)}")
    print("\nFirst 5 testing images:", test_imgs[:5])
    print("First 5 testing masks:", test_masks[:5])
    
    # Check file extensions
    train_img_ext = set(os.path.splitext(f)[1] for f in train_imgs)
    train_mask_ext = set(os.path.splitext(f)[1] for f in train_masks)
    print("\nFile Extensions:")
    print(f"Training images: {train_img_ext}")
    print(f"Training masks: {train_mask_ext}")
    
    # Check matching pairs
    train_img_bases = set(os.path.splitext(f)[0] for f in train_imgs)
    train_mask_bases = set(os.path.splitext(f)[0] for f in train_masks)
    matching_pairs = train_img_bases & train_mask_bases
    
    print(f"\nMatching pairs found: {len(matching_pairs)}")
    if len(matching_pairs) == 0:
        print("WARNING: No matching image-mask pairs found!")
        print("\nExample image names (without extension):", list(train_img_bases)[:5])
        print("Example mask names (without extension):", list(train_mask_bases)[:5])

check_dataset_structure()

In [ ]:
def verify_file_extensions():
    """Check file extensions in both training and testing directories"""
    for file in os.listdir(TRAIN_MASK_PATH):
        if not file.endswith('.jpg'):  # veya .png
            print(f"Warning: Inconsistent file extension found: {file}")
            
verify_file_extensions()

In [ ]:
def verify_paths():
    """Verify all data paths and print detailed information"""
    print("Verifying data directories...")
    
    paths = {
        'Training Images': TRAIN_IMG_PATH,
        'Training Masks': TRAIN_MASK_PATH,
        'Testing Images': TEST_IMG_PATH,
        'Testing Masks': TEST_MASK_PATH
    }
    
    for name, path in paths.items():
        print(f"\n{name} path: ", end='')
        if os.path.exists(path):
            files = sorted(os.listdir(path))
            print("OK")
            print(f"Path: {path}")
            print(f"Files found: {len(files)}")
            print(f"First few files: {files[:3]}")
            print("-" * 50)
        else:
            print("NOT FOUND")
            print(f"Path {path} does not exist!")
            raise ValueError(f"Directory not found: {path}")
    
    print("All data directories verified successfully!")

verify_paths()

# Data loading

In [ ]:
def load_data(img_path, mask_path):
    """Load and preprocess images and masks"""
    images = []
    masks = []
    errors = []
    
    print(f"\nLoading data from:")
    print(f"Images: {img_path}")
    print(f"Masks: {mask_path}")
    
    # Görüntü dosyalarını listele
    img_files = sorted([f for f in os.listdir(img_path) if f.endswith('.jpg')])
    
    for img_file in tqdm(img_files, desc="Loading images"):
        try:
            # Görüntü dosyası yolu
            img_filepath = os.path.join(img_path, img_file)
            
            # Mask dosyası için hem .jpg hem .png uzantısını dene
            base_name = os.path.splitext(img_file)[0]
            mask_filepath_jpg = os.path.join(mask_path, f"{base_name}.jpg")
            mask_filepath_png = os.path.join(mask_path, f"{base_name}.png")
            
            # Önce .png sonra .jpg olarak kontrol et
            if os.path.exists(mask_filepath_png):
                mask_filepath = mask_filepath_png
            elif os.path.exists(mask_filepath_jpg):
                mask_filepath = mask_filepath_jpg
            else:
                errors.append(f"Mask not found: {mask_filepath_png} or {mask_filepath_jpg}")
                continue
            
            # Görüntüleri yükle
            img = cv2.imread(img_filepath)
            if img is None:
                errors.append(f"Failed to load image: {img_filepath}")
                continue
                
            mask = cv2.imread(mask_filepath, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                errors.append(f"Failed to load mask: {mask_filepath}")
                continue
            
            # BGR'den RGB'ye dönüştür
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Boyutlandırma
            img = cv2.resize(img, IMAGE_SIZE)
            mask = cv2.resize(mask, IMAGE_SIZE)
            
            # Normalizasyon öncesi görüntü kontrolü
            print(f"\nBefore normalization for {img_file}:")
            print(f"Image range: [{img.min()}, {img.max()}]")
            print(f"Mask range: [{mask.min()}, {mask.max()}]")
            
            
            # Normalizasyon
            img = img / 255.0
            mask = (mask > 128).astype(np.float32)  # Binary threshold ekle
            
            print(f"After normalization for {img_file}:")
            print(f"Image range: [{img.min():.3f}, {img.max():.3f}]")
            print(f"Mask range: [{mask.min():.3f}, {mask.max():.3f}]")
            
            # Mask'ı 3. boyut olarak ekle
            mask = np.expand_dims(mask, axis=-1)
            
            images.append(img)
            masks.append(mask)
            
        except Exception as e:
            errors.append(f"Error processing {img_file}: {str(e)}")
            continue
    
    if errors:
        print("\nProcessing Errors:")
        for error in errors[:10]:  # İlk 10 hatayı göster
            print(f"- {error}")
        if len(errors) > 10:
            print(f"... and {len(errors) - 10} more errors")
    
    if not images:
        print("\nDetailed directory information:")
        print(f"Images directory contents: {os.listdir(img_path)[:5]}")
        print(f"Masks directory contents: {os.listdir(mask_path)[:5]}")
        raise ValueError("No valid image-mask pairs found! Check your data directories and file names.")
    
    print(f"\nSuccessfully loaded {len(images)} image-mask pairs")
    return np.array(images), np.array(masks)

train_images, train_masks = load_data(TRAIN_IMG_PATH, TRAIN_MASK_PATH)
test_images, test_masks = load_data(TEST_IMG_PATH, TEST_MASK_PATH)

print(f"\nTraining samples: {len(train_images)}")
print(f"Testing samples: {len(test_images)}")

# Resplit Data

In [ ]:
images = np.concatenate((train_images, test_images), axis=0)
masks = np.concatenate((train_masks, test_masks), axis=0)

print(images.shape)
print(masks.shape)

train_images, test_images, train_masks, test_masks = train_test_split(images, masks, test_size=0.2, random_state=42)

print(train_images.shape)
print(test_images.shape)
print(train_masks.shape)
print(test_masks.shape)

# Verify data quality

In [ ]:
def verify_data_quality(images, masks):
    """Verify data quality and print statistics"""
    print("\nData Quality Check:")
    print(f"Image array shape: {images.shape}")
    print(f"Mask array shape: {masks.shape}")
    print(f"Image value range: [{images.min():.3f}, {images.max():.3f}]")
    print(f"Mask value range: [{masks.min():.3f}, {masks.max():.3f}]")
    
    # Mask değerlerinin dağılımı
    mask_values = masks.flatten()
    unique_values = np.unique(mask_values)
    print(f"Unique mask values: {unique_values}")
    
    # Pozitif/negatif piksel oranı
    positive_pixels = np.sum(mask_values > 0.5)
    total_pixels = mask_values.size
    print(f"Positive pixel ratio: {positive_pixels/total_pixels:.3f}")

verify_data_quality(train_images, train_masks)
verify_data_quality(test_images, test_masks)

# Show Image Samples

In [ ]:
def display_image_grid(images, num_images=16, save_path='Project_Outputs/training_plots/image_grid'):
    """Display and save a grid of images"""
    os.makedirs(save_path, exist_ok=True)
    
    # Calculate grid dimensions
    grid_size = int(np.ceil(np.sqrt(num_images)))
    
    plt.figure(figsize=(10, 10))
    for idx in range(min(num_images, len(images))):
        plt.subplot(grid_size, grid_size, idx + 1)
        plt.imshow(images[idx])
        plt.axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{save_path}/image_grid.png')
    plt.show()
    plt.close()

display_image_grid(train_images)

# Verifying image-mask pairs

In [ ]:
def verify_image_mask_pairs(images, masks, num_samples=5, save_path='Project_Outputs/training_plots/verification'):
    """Verify and visualize image-mask pairs"""
    os.makedirs(save_path, exist_ok=True)
    
    for idx in range(min(num_samples, len(images))):
        plt.figure(figsize=(12, 4))
        
        # Original Image
        plt.subplot(141)
        plt.imshow(images[idx])
        plt.title(f'Image {idx}')
        plt.axis('off')
        
        # Mask
        plt.subplot(142)
        plt.imshow(masks[idx, ..., 0], cmap='gray')
        plt.title(f'Mask {idx}')
        plt.axis('off')
        
        # Overlay
        plt.subplot(143)
        overlay = images[idx].copy()
        overlay[masks[idx, ..., 0] > 0.5] = [1, 0, 0]  # Red overlay for mask
        plt.imshow(overlay)
        plt.title('Overlay')
        plt.axis('off')

        plt.subplot(144)
        plt.imshow(images[idx])
        plt.imshow(masks[idx, ..., 0], alpha=0.5, cmap='gray') # Maskeyi görüntü üzerine şeffaf olarak çiz
        plt.title("image&mask")
        plt.axis("off")
        
        plt.savefig(f'{save_path}/pair_verification_{idx}.png')
        plt.show()
        plt.close()
        
    print(f"Saved {num_samples} verification images to {save_path}")

verify_image_mask_pairs(train_images, train_masks)

In [ ]:
def display_mask_distribution(masks, save_path='Project_Outputs/training_plots/mask_distribution'):
    """Display and save mask value distribution"""
    os.makedirs(save_path, exist_ok=True)
    
    mask_values = masks.flatten()
    
    plt.figure(figsize=(10, 6))
    plt.hist(mask_values, bins=50, density=True)
    plt.title('Mask Value Distribution')
    plt.xlabel('Pixel Values')
    plt.ylabel('Density')
    plt.savefig(f'{save_path}/mask_distribution.png')
    plt.show()
    plt.close()
display_mask_distribution(train_masks)

# Data Augmentation

In [ ]:
def data_augmentation():
    """Enhanced data augmentation pipeline for camouflage object detection"""
    return A.Compose([
        A.RandomRotate90(p=0.8),
        A.HorizontalFlip(p=0.7),
        A.VerticalFlip(p=0.7),
        #A.Perspective(scale=(0.05, 0.15), p=0.6),
    ],
        additional_targets={'mask': 'image'}
    )
data_augmentation = data_augmentation()

In [ ]:
def augment_data(training_images, training_mask, num_augmentations=1):
    augmented_images = []
    augmented_masks = []

    # Uzunlukları kontrol et
    if len(training_images) != len(training_mask):
        print("HATA: training_images ve training_mask listelerinin uzunlukları farklı!")
        # Eşitleme stratejisi:
        # 1. En kısa olanın uzunluğuna göre kesmek
        min_len = min(len(training_images), len(training_mask))
        training_images = training_images[:min_len]
        training_mask = training_mask[:min_len]
        # 2. Eksik olanları tamamlamak (örneğin, boş maskelerle)
        # 3. Hata verip durdurmak (veri tutarsızlığını işaret eder)
        # Burada en basit çözüm olan kesme yöntemini uyguladım.
        # Veri setine bağlı olarak diğer stratejiler daha uygun olabilir.
        print("Listeler eşitlendi. Yeni uzunluk:", min_len)
    
    for i in range(len(training_images)):
        image = training_images[i]
        mask = training_mask[i]

        for _ in range(num_augmentations):  # Her resim için birden fazla artırma yapabilirsiniz
            # Veri tipini kontrol et ve gerekirse dönüştür

            #mask = np.stack([mask] * 3, axis=-1) #maskı 3 kanala dönüştür
            
            if image.dtype == np.float64:
                image = (image * 255).astype(np.uint8)  # float64 ise 0-1 aralığına getir ve uint8'e dönüştür
            if mask.dtype == np.float64:
                mask = (mask * 255).astype(np.uint8)  # float64 ise 0-1 aralığına getir ve uint8'e dönüştür
            
            augmented = data_augmentation(image=image, mask=mask)
            augmented_image = augmented['image']
            augmented_mask = augmented['mask']

            #augmented_mask = np.mean(augmented_mask, axis=2) #maskı tekrar 1 kanala dönüştür

            augmented_images.append(augmented_image)
            augmented_masks.append(augmented_mask)
            
    
    augmented_images = np.array(augmented_images)
    augmented_masks = np.array(augmented_masks)

    return augmented_images, augmented_masks


augmented_images, augmented_masks = augment_data(train_images, train_masks, num_augmentations=3)

# Artırılmış veriyi orijinal verilere ekleme 
train_images = np.concatenate((train_images, augmented_images), axis=0)
train_masks = np.concatenate((train_masks, augmented_masks), axis=0)

In [ ]:
def display_augmentation_examples(image, mask, aug_images, aug_masks, num_examples=5, 
                                save_path='Project_Outputs/training_plots/augmentation'):
    """Display and save augmentation examples"""
    os.makedirs(save_path, exist_ok=True)
    
    # Convert image and mask to uint8
    image = (image * 255).astype(np.uint8)
    mask = (mask * 255).astype(np.uint8)
    
    # Ensure mask has the same number of channels as the image
    if mask.shape[-1] != image.shape[-1]:
        mask = np.repeat(mask, image.shape[-1], axis=-1)
    
    plt.figure(figsize=(20, 8))
    # Original
    plt.subplot(2, num_examples + 1, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(2, num_examples + 1, num_examples + 2)
    plt.imshow(mask[..., 0], cmap='gray')  # Show only one channel of the mask
    plt.title('Original Mask')
    plt.axis('off')
    
    # Augmented examples
    for idx in range(num_examples):
        aug_image = (aug_images[idx] * 255).astype(np.uint8)
        aug_mask = (aug_masks[idx] * 255).astype(np.uint8)

        plt.subplot(2, num_examples + 1, idx + 2)
        plt.imshow(aug_image)
        plt.title(f'Augmented Image {idx+1}')
        plt.axis('off')
        
        plt.subplot(2, num_examples + 1, idx + num_examples + 3)
        plt.imshow(aug_mask[..., 0], cmap='gray')  # Show only one channel of the mask
        plt.title(f'Augmented Mask {idx+1}')
        plt.axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{save_path}/augmentation_examples.png')
    plt.show()
    plt.close()

display_augmentation_examples(train_images[0], train_masks[0], augmented_images, augmented_masks)

In [ ]:
#ikili maskeye dönüştürme
def binary_mask(mask, threshold):
  binary_mask = np.where(mask >= threshold, 1, 0)
  return binary_mask

train_masks= binary_mask(train_masks,0.5)
test_masks= binary_mask(test_masks,0.5)

# Model Creation

In [ ]:
def tp_tn_fp_fn(y_true, y_pred, threshold=0.5):
    """
    TP, TN, FP ve FN değerlerini hesaplar.
    ... (Önceki fonksiyonun tanımı)
    """
    y_pred_binary = tf.cast(y_pred > threshold, tf.int32)
    tp = tf.reduce_sum(tf.cast((y_true == 1) & (y_pred_binary == 1), tf.int32))
    tn = tf.reduce_sum(tf.cast((y_true == 0) & (y_pred_binary == 0), tf.int32))
    fp = tf.reduce_sum(tf.cast((y_true == 0) & (y_pred_binary == 1), tf.int32))
    fn = tf.reduce_sum(tf.cast((y_true == 1) & (y_pred_binary == 0), tf.int32))
    return tp, tn, fp, fn

#Dice katsayısı
def dice_coefficient(y_true, y_pred, threshold=0.5):

    tp, tn, fp, fn = tp_tn_fp_fn(y_true, y_pred, threshold)
    
 # tp, fp, ve fn değerlerini float32'ye dönüştür
    tp = tf.cast(tp, tf.float32)
    fp = tf.cast(fp, tf.float32)
    fn = tf.cast(fn, tf.float32)

    # Dice katsayısını hesapla
    dice = (2 * tp ) / (2 * tp + fp + fn )
    return dice

# F1 skoru
def f1_score(y_true, y_pred, threshold=0.5):
    tp, tn, fp, fn = tp_tn_fp_fn(y_true, y_pred, threshold)

    # Değerleri float32'ye dönüştür
    tp = tf.cast(tp, tf.float32)
    fp = tf.cast(fp, tf.float32)
    fn = tf.cast(fn, tf.float32)

    precision = tp / (tp + fp )
    recall = tp / (tp + fn )

    f1 = 2 * (precision * recall) / (precision + recall )
    return f1


#Matthews Korelasyon Katsayısı (MCC) metriği
def mcc(y_true, y_pred, threshold=0.5):

    tp, tn, fp, fn = tp_tn_fp_fn(y_true, y_pred, threshold)

    # tp, tn, fp, ve fn değerlerini float32'ye dönüştür
    tp = tf.cast(tp, tf.float32)
    tn = tf.cast(tn, tf.float32)
    fp = tf.cast(fp, tf.float32)
    fn = tf.cast(fn, tf.float32)

    # MCC hesapla
    numerator = (tp * tn) - (fp * fn)
    denominator = tf.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))

    # Sıfıra bölme hatasını engelle
    mcc_score = tf.cond(tf.equal(denominator, 0),
                                lambda: tf.constant(0.0, dtype=tf.float32),
                                lambda: numerator / denominator)
    return mcc_score


metric_funcs=[
    'accuracy',
    krs.metrics.MeanIoU(num_classes=2,name="mean_iou"),
    krs.metrics.IoU(num_classes=2, target_class_ids=[0],name="iou"),
    dice_coefficient,
    f1_score,
    krs.metrics.Recall(name="recall"),
    krs.metrics.Precision(name="precision"),
    krs.metrics.AUC(name="auc"),
    mcc,
    ]

# List of metric/loss names
metrics = ['loss', 'iou',"mean_iou", 'accuracy',"dice_coefficient","recall","precision","f1_score","mcc"]

In [ ]:
# Model compilation

nb_filter = [32, 64, 128, 256]

# --- Attention Gate Helper Function ---
def attention_gate(x, g, inter_channel, name):
    """
    Attention Gate mechanism.

    Args:
        x (tensor): Input features from the encoder skip connection.
        g (tensor): Gating signal from the decoder path (upsampled).
        inter_channel (int): Number of intermediate channels.
        name (str): Base name for the layers.

    Returns:
        tensor: Attended features (output of the attention gate).
    """
    # Theta^T * x_ij + Phi^T * gating_signal + bias_g
    theta_x = layers.Conv2D(inter_channel, kernel_size=(1, 1), strides=(1, 1), padding='same', use_bias=False, name=name+'_theta_x')(x)
    phi_g = layers.Conv2D(inter_channel, kernel_size=(1, 1), strides=(1, 1), padding='same', use_bias=True, name=name+'_phi_g')(g)

    # Element-wise sum
    f = layers.Activation('relu')(layers.add([theta_x, phi_g], name=name+'_add'))

    # Psi^T * f + bias_psi
    psi_f = layers.Conv2D(1, kernel_size=(1, 1), strides=(1, 1), padding='same', use_bias=True, name=name+'_psi_f')(f)

    # Sigmoid activation to get attention coefficients (alpha)
    alpha = layers.Activation('sigmoid', name=name+'_alpha')(psi_f)

    # Multiply attention coefficients with input features x
    # x_ij * alpha_ij
    output = layers.multiply([x, alpha], name=name+'_attended_x')

    return output

# --- Helper Function for Convolutional Blocks ---
# (Dropout oranı parametrik yapıldı)
def conv_block(input_tensor, num_filters, kernel_size=(3, 3), activation='relu', kernel_initializer='he_normal', padding='same', use_dropout=True, dropout_rate=0.5):
    """Creates a block of two Conv2D layers with optional Dropout."""
    x = layers.Conv2D(num_filters, kernel_size, kernel_initializer=kernel_initializer, padding=padding)(input_tensor)
    x = layers.BatchNormalization()(x) # Batch Normalization eklendi (genellikle faydalıdır)
    x = layers.Activation(activation)(x)
    if use_dropout and dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)

    x = layers.Conv2D(num_filters, kernel_size, kernel_initializer=kernel_initializer, padding=padding)(x)
    x = layers.BatchNormalization()(x) # Batch Normalization eklendi
    x = layers.Activation(activation)(x)
    if use_dropout and dropout_rate > 0:
        x = layers.Dropout(dropout_rate)(x)
    return x

# --- Build ResNet50 Backbone U-Net++ with Attention Gates ---
def build_resnet50_attention_unet_plus_plus(input_shape, num_classes=1, dropout_rate=0.5):
    """
    Builds a U-Net++ model with a ResNet50 encoder backbone and Attention Gates.

    Args:
        input_shape (tuple): Shape of the input image (height, width, channels).
        num_classes (int): Number of output classes (for segmentation, usually 1 for binary).
        dropout_rate (float): Dropout rate to apply in convolutional blocks.

    Returns:
        tf.keras.Model: The constructed Attention U-Net++ model.
    """
    inputs = layers.Input(input_shape)

    # Load ResNet50 backbone
    base_model = ResNet50(include_top=False, weights='imagenet', input_tensor=inputs)
    base_model.trainable = True # Fine-tuning için

    # --- Extract Skip Connection Layers from ResNet50 ---
    skip_1_out = base_model.get_layer('conv1_relu').output         # X_00 ~ H/2
    skip_2_out = base_model.get_layer('conv2_block3_out').output   # X_10 ~ H/4
    skip_3_out = base_model.get_layer('conv3_block4_out').output   # X_20 ~ H/8
    skip_4_out = base_model.get_layer('conv4_block6_out').output   # X_30 ~ H/16
    bridge_out = base_model.get_layer('conv5_block3_out').output   # X_40 ~ H/32 (Deepest)

    # --- U-Net++ Decoder with Attention Gates ---
    # (X_ij notation refers to the U-Net++ paper diagram)

    # Level 1 (Encoder Features)
    X_00 = skip_1_out

    # Level 2
    X_10 = skip_2_out
    Up_10_to_01 = layers.Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), padding='same', name='up_10_to_01')(X_10)
    # --- Attention Gate AG(X_00, Up_10_to_01) ---
    Att_X_00_from_10 = attention_gate(X_00, Up_10_to_01, inter_channel=nb_filter[0] // 2, name='ag_00_from_10')
    Merge_01 = layers.concatenate([Up_10_to_01, Att_X_00_from_10], axis=-1, name='merge_01')
    X_01 = conv_block(Merge_01, nb_filter[0], dropout_rate=dropout_rate)

    # Level 3
    X_20 = skip_3_out
    Up_20_to_11 = layers.Conv2DTranspose(nb_filter[1], (2, 2), strides=(2, 2), padding='same', name='up_20_to_11')(X_20)
    # --- Attention Gate AG(X_10, Up_20_to_11) ---
    Att_X_10_from_20 = attention_gate(X_10, Up_20_to_11, inter_channel=nb_filter[1] // 2, name='ag_10_from_20')
    Merge_11 = layers.concatenate([Up_20_to_11, Att_X_10_from_20], axis=-1, name='merge_11')
    X_11 = conv_block(Merge_11, nb_filter[1], dropout_rate=dropout_rate)

    Up_11_to_02 =layers.Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), padding='same', name='up_11_to_02')(X_11)
    # --- Attention Gate AG(X_00, Up_11_to_02) ---
    Att_X_00_from_11 = attention_gate(X_00, Up_11_to_02, inter_channel=nb_filter[0] // 2, name='ag_00_from_11')
    Merge_02 = layers.concatenate([Up_11_to_02, Att_X_00_from_11, X_01], axis=-1, name='merge_02') # X_01'e AG uygulanmıyor
    X_02 = conv_block(Merge_02, nb_filter[0], dropout_rate=dropout_rate)

    # Level 4
    X_30 = skip_4_out
    Up_30_to_21 = layers.Conv2DTranspose(nb_filter[2], (2, 2), strides=(2, 2), padding='same', name='up_30_to_21')(X_30)
    # --- Attention Gate AG(X_20, Up_30_to_21) ---
    Att_X_20_from_30 = attention_gate(X_20, Up_30_to_21, inter_channel=nb_filter[2] // 2, name='ag_20_from_30')
    Merge_21 = layers.concatenate([Up_30_to_21, Att_X_20_from_30], axis=-1, name='merge_21')
    X_21 = conv_block(Merge_21, nb_filter[2], dropout_rate=dropout_rate)

    Up_21_to_12 = layers.Conv2DTranspose(nb_filter[1], (2, 2), strides=(2, 2), padding='same', name='up_21_to_12')(X_21)
    # --- Attention Gate AG(X_10, Up_21_to_12) ---
    Att_X_10_from_21 = attention_gate(X_10, Up_21_to_12, inter_channel=nb_filter[1] // 2, name='ag_10_from_21')
    Merge_12 = layers.concatenate([Up_21_to_12, Att_X_10_from_21, X_11], axis=-1, name='merge_12') # X_11'e AG uygulanmıyor
    X_12 = conv_block(Merge_12, nb_filter[1], dropout_rate=dropout_rate)

    Up_12_to_03 = layers.Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), padding='same', name='up_12_to_03')(X_12)
    # --- Attention Gate AG(X_00, Up_12_to_03) ---
    Att_X_00_from_12 = attention_gate(X_00, Up_12_to_03, inter_channel=nb_filter[0] // 2, name='ag_00_from_12')
    Merge_03 = layers.concatenate([Up_12_to_03, Att_X_00_from_12, X_01, X_02], axis=-1, name='merge_03') # X_01, X_02'ye AG uygulanmıyor
    X_03 = conv_block(Merge_03, nb_filter[0], dropout_rate=dropout_rate)

    # Level 5 (Bridge)
    X_40 = bridge_out
    Up_40_to_31 = layers.Conv2DTranspose(nb_filter[3], (2, 2), strides=(2, 2), padding='same', name='up_40_to_31')(X_40)
    # --- Attention Gate AG(X_30, Up_40_to_31) ---
    Att_X_30_from_40 = attention_gate(X_30, Up_40_to_31, inter_channel=nb_filter[3] // 2, name='ag_30_from_40')
    Merge_31 =layers.concatenate([Up_40_to_31, Att_X_30_from_40], axis=-1, name='merge_31')
    X_31 = conv_block(Merge_31, nb_filter[3], dropout_rate=dropout_rate)

    Up_31_to_22 = layers.Conv2DTranspose(nb_filter[2], (2, 2), strides=(2, 2), padding='same', name='up_31_to_22')(X_31)
    # --- Attention Gate AG(X_20, Up_31_to_22) ---
    Att_X_20_from_31 = attention_gate(X_20, Up_31_to_22, inter_channel=nb_filter[2] // 2, name='ag_20_from_31')
    Merge_22 = layers.concatenate([Up_31_to_22, Att_X_20_from_31, X_21], axis=-1, name='merge_22') # X_21'e AG uygulanmıyor
    X_22 = conv_block(Merge_22, nb_filter[2], dropout_rate=dropout_rate)

    Up_22_to_13 = layers.Conv2DTranspose(nb_filter[1], (2, 2), strides=(2, 2), padding='same', name='up_22_to_13')(X_22)
    # --- Attention Gate AG(X_10, Up_22_to_13) ---
    Att_X_10_from_22 = attention_gate(X_10, Up_22_to_13, inter_channel=nb_filter[1] // 2, name='ag_10_from_22')
    Merge_13 = layers.concatenate([Up_22_to_13, Att_X_10_from_22, X_11, X_12], axis=-1, name='merge_13') # X_11, X_12'ye AG uygulanmıyor
    X_13 = conv_block(Merge_13, nb_filter[1], dropout_rate=dropout_rate)

    Up_13_to_04 = layers.Conv2DTranspose(nb_filter[0], (2, 2), strides=(2, 2), padding='same', name='up_13_to_04')(X_13)
    # --- Attention Gate AG(X_00, Up_13_to_04) ---
    Att_X_00_from_13 = attention_gate(X_00, Up_13_to_04, inter_channel=nb_filter[0] // 2, name='ag_00_from_13')
    Merge_04 = layers.concatenate([Up_13_to_04, Att_X_00_from_13, X_01, X_02, X_03], axis=-1, name='merge_04') # X_01, X_02, X_03'e AG uygulanmıyor
    X_04 = conv_block(Merge_04, nb_filter[0], dropout_rate=dropout_rate)

    final_upsample = layers.UpSampling2D(size=(2, 2), interpolation='bilinear', name='final_upsample')(X_04)
    # Artık final_upsample boyutu (None, 256, 256, nb_filter[0])

    output_activation = 'sigmoid' if num_classes == 1 else 'softmax'
    # Final 1x1 Conv katmanını final_upsample üzerine uygula
    outputs = layers.Conv2D(num_classes, (1, 1), activation=output_activation, name='final_output')(final_upsample)
    # outputs boyutu şimdi (None, 256, 256, num_classes) olmalı

    # --- Create Model ---
    model = Model(inputs=[inputs], outputs=[outputs], name='ResNet50_Attention_UNet_Plus_Plus')

    return model
    
model = build_resnet50_attention_unet_plus_plus(
    input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3),
    num_classes=1,
    dropout_rate=DROPOUT_RATE
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=LR,
        beta_1=0.9,
        beta_2=0.999,
        epsilon=1e-07
    ),
    loss="binary_focal_crossentropy",
    metrics=metric_funcs
)

# Model özeti
model.summary()

# Model Training

In [ ]:
from sklearn.model_selection import KFold
kfold = KFold(n_splits=KFOLD_NUM, shuffle=True, random_state=42)

scorelists=[]
egitim_baslangic = time.time()

for kfold_index, (train, test) in zip(range(KFOLD_NUM), kfold.split(images, masks)):
    
    # Training with modified parameters
    print(f"\nStarting model training...\n K-Fold={kfold_index+1}/{KFOLD_NUM}")
    history = model.fit(
        images[train], masks[train],
        validation_split=0.2,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        callbacks=[
            tf.keras.callbacks.ModelCheckpoint(
                'Project_Outputs/model_outputs/best_model.keras',
                save_best_only=True,
                monitor='val_iou_metric',
                mode='max',
                verbose=1
            ),

            tf.keras.callbacks.CSVLogger(
                'Project_Outputs/model_outputs/training_log.csv', 
                separator=',', 
                append=False),
            
            #tf.keras.callbacks.TensorBoard(
                #log_dir='Project_Outputs/logs',
                #histogram_freq=1,
                #write_graph=True,
                #write_images=True,
                #update_freq='epoch',
                #profile_batch=2, 
                #embeddings_freq=1)
        ],
        verbose=1
    )
    scores = model.evaluate(images[test], masks[test], return_dict=True, verbose=1)
    scorelists.append(scores)

egitim_bitis = time.time()
egitim_suresi = egitim_bitis - egitim_baslangic
print(f"Eğitim süresi: {egitim_suresi:.4f} saniye")

In [ ]:
# Tahmin süresini ölçme
tahmin_baslangic = time.time()
tahminler = model.predict(np.expand_dims(images[np.random.randint(0, len(images))], axis=0))
tahmin_bitis = time.time()
tahmin_suresi = tahmin_bitis - tahmin_baslangic
print(f"Tahmin süresi: {tahmin_suresi:.4f} saniye")

# Training visualizations and metrics

In [ ]:
#KFold metrics
sonuclar = {}
for anahtar in scorelists[0].keys():
    degerler = [d[anahtar] for d in scorelists]
    ortalama = np.mean(degerler)
    standart_sapma = np.std(degerler)
    sonuclar[anahtar] = {"ortalama": ortalama, "standart_sapma": standart_sapma}

df_sonuclar = pd.DataFrame.from_dict(sonuclar, orient='index')
# DataFrame'deki sayısal sütunları belirli bir duyarlılıkla biçimlendirme
def format_numeric_columns(df, precision=4):
    for col in df.columns:
        if df[col].dtype == 'float64' or df[col].dtype == 'float32':
            df[col] = df[col].map(lambda x: f"{x:.{precision}f}")
    return df
kflod_metrics_df = format_numeric_columns(df_sonuclar)

kflod_metrics_df.to_csv('Project_Outputs/model_outputs/KFold_metrics.csv')
print(df_sonuclar)


In [ ]:
def plot_metrics(history, epochs, save_dir='Project_Outputs/training_plots'):
    """
    Plot and save comprehensive training metrics with best epoch indication, only for loss.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Plot each metric
    for metric in metrics:
        plt.figure(figsize=(35, 25))
        plt.plot(history.history[metric], label=f'Training {metric}')
        plt.plot(history.history[f'val_{metric}'], label=f'Validation {metric}')
        
        if 'loss' in metric:
            best_epoch = np.argmin(history.history[f'val_{metric}'])
            best_value = history.history[f'val_{metric}'][best_epoch]
            
            plt.plot(best_epoch, best_value, marker='o', color='red', label=f'Best Epoch ({best_epoch+1})')
        
        plt.title(f'Model {metric}')
        plt.xticks(rotation=45)
        plt.xlabel('Epoch')
        plt.ylabel(metric)
        plt.legend()
        plt.grid(True)
        plt.savefig(os.path.join(save_dir, f'{metric}_history.png'))
        plt.show()
        plt.close()
plot_metrics(history, EPOCHS)

# Generate predictions

In [ ]:
test_predictions = model.predict(test_images, verbose=1)
train_predictions = model.predict(train_images, verbose=1)



test_predictions= binary_mask(test_predictions,0.5)
train_predictions= binary_mask(train_predictions,0.5)
#train_masks= binary_mask(train_masks,0.5)

# Compare Predictions

In [ ]:
def display_prediction_comparison(images, masks, predictions, num_samples=5, 
                                save_path='Project_Outputs/training_plots/predictions'):
    """Display and save comparison of predictions"""
    os.makedirs(save_path, exist_ok=True)
    
    for idx in range(num_samples):
        plt.figure(figsize=(15, 5))
        
        # Original Image
        plt.subplot(1, 4, 1)
        plt.imshow(images[idx])
        plt.title('Original Image')
        plt.axis('off')
        
        # True Mask
        plt.subplot(1, 4, 2)
        plt.imshow(masks[idx, ..., 0], cmap='gray')
        plt.title('True Mask')
        plt.axis('off')
        
        # Predicted Mask
        plt.subplot(1, 4, 3)
        plt.imshow(predictions[idx, ..., 0], cmap='gray')
        plt.title('Predicted Mask')
        plt.axis('off')

        plt.subplot(1,4,4)
        plt.imshow(images[idx])
        plt.imshow(predictions[idx, ..., 0], alpha=0.5, cmap='gray') # Maskeyi görüntü üzerine şeffaf olarak çiz
        plt.title("image&mask")
        plt.axis("off")
        
        plt.tight_layout()
        plt.savefig(f'{save_path}/prediction_comparison_{idx+1}.png')
        plt.show()
        plt.close()

display_prediction_comparison(
    test_images,
    test_masks,
    test_predictions
)

# Generate and save ROC-AUC curve

In [ ]:
# Eğitim kümesi için ROC eğrisi ve AUC
fpr_train, tpr_train, thresholds_train = roc_curve(train_masks.ravel(), train_predictions.ravel())
auc_train = auc(fpr_train, tpr_train)

# Test kümesi için ROC eğrisi ve AUC
fpr_test, tpr_test, thresholds_test = roc_curve(test_masks.ravel(), test_predictions.ravel())
auc_test = auc(fpr_test, tpr_test)

# ROC eğrilerini çizin
plt.figure(figsize=(8, 6))
plt.plot(fpr_train, tpr_train, label=f'Eğitim ROC Eğrisi (AUC = {auc_train:.2f})')
plt.plot(fpr_test, tpr_test, label=f'Test ROC Eğrisi (AUC = {auc_test:.2f})')
plt.plot([0, 1], [0, 1], 'k--')  # Rastgele tahmin çizgisi
plt.xlabel('Yanlış Pozitif Oranı (FPR)')
plt.ylabel('Doğru Pozitif Oranı (TPR)')
plt.title('ROC Eğrisi')
plt.legend(loc='lower right')
plt.savefig('Project_Outputs/training_plots/roc_curve.png')
plt.show()
plt.close()

# Save final model

In [ ]:
print("\nSaving final model...")
model.save('Project_Outputs/model_outputs/final_model.keras')

# Final evaluation

In [ ]:
import numpy as np

final_metrics = model.evaluate(images[test], masks[test], return_dict=True, verbose=1)

# Convert final_metrics to a Pandas DataFrame
final_metrics_df = pd.DataFrame([final_metrics],index=["Final Evaluation"]).T

final_metrics_df = format_numeric_columns(final_metrics_df)

final_metrics_df.to_csv('Project_Outputs/model_outputs/final_metrics.csv')
print(final_metrics_df)

In [ ]:
comparison_df = pd.concat([kflod_metrics_df['standart_sapma'], kflod_metrics_df['ortalama'], final_metrics_df[final_metrics_df.columns[0]]], axis=1)
comparison_df.columns = ["KFold Std. Sapma",'KFold Ortalama', 'Final Evaluation']

comparison_df.to_csv('Project_Outputs/model_outputs/model_comparison.csv')
print(comparison_df)